# 🎯 WavLM Word-Level Feature Extraction — Colab T4 GPU
**Goal:** Extract WavLM+prosody features for 976 EMNLP videos on Colab T4 GPU
**Auto-resume:** Saves checkpoints to Google Drive — survives timeouts
**Runtime:** ~22 hours total (multiple sessions)

**What this does:**
1. Mount Drive + install deps
2. Load WavLM on GPU
3. Find all videos with EMNLP labels
4. Auto-resume from checkpoint
5. Extract WavLM 768-dim + prosody 23-dim = 791-dim per word
6. Save features to Drive

In [ ]:
# Cell 1: Setup + Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, json, time, warnings
warnings.filterwarnings('ignore')

BASE = '/content/drive/MyDrive/standup4ai'
FEAT_DIR = f'{BASE}/word_level_features'
LABEL_BASE = f'{BASE}/seq-Standup4AI/dataset'
AUDIO_BASE = f'{BASE}'

os.makedirs(FEAT_DIR, exist_ok=True)

# Check GPU
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'GPU: {device}')
if device.type != 'cuda':
    print('⚠️ WARNING: No GPU! Go to Runtime → Change runtime type → T4 GPU')

# Install
import subprocess
subprocess.run(['pip', 'install', '-q', 'soundfile', 'librosa', 'transformers'], capture_output=True)
print('✓ Dependencies installed')

In [ ]:
# Cell 2: Load WavLM on GPU
from transformers import AutoModel
import torch

print('Loading WavLM on GPU...')
wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
wavlm.to(device)
wavlm.eval()
print(f'✓ WavLM loaded on {device}')

# Test
import numpy as np
test = torch.randn(1, 16000).to(device)
with torch.no_grad():
    out = wavlm(test)
print(f'✓ WavLM test OK: {out.last_hidden_state.shape}')

In [ ]:
# Cell 3: Feature extraction functions
import librosa
import numpy as np
import soundfile as sf
import pandas as pd
import ast

SR_AUDIO = 16000  # WavLM expects 16kHz

def load_audio(audio_path):
    """Load audio with soundfile (fast), resample to 16kHz."""
    y, sr = sf.read(audio_path, dtype='float32')
    if len(y.shape) > 1: y = y.mean(axis=1)
    if sr != SR_AUDIO:
        y = librosa.resample(y, orig_sr=sr, target_sr=SR_AUDIO)
    return y

def extract_23dim_prosody(y, t0, t1):
    """Extract 23-dim prosody for word segment [t0, t1] seconds."""
    s, e = int(t0 * 22050), min(int(t1 * 22050), len(y))
    chunk = y[s:e]
    if len(chunk) < 0.05 * 22050: return None
    
    # F0 (5 dims)
    try:
        f0, voiced, _ = librosa.pyin(chunk, fmin=50, fmax=500, sr=22050)
        f0c = f0[~np.isnan(f0)]
        v = voiced[~np.isnan(f0)]
        f0_feats = [np.mean(f0c), np.std(f0c), np.max(f0c), np.min(f0c), np.mean(v)] if len(f0c) > 0 else [0]*5
    except: f0_feats = [0]*5
    
    # Energy (5 dims)
    hop = 512
    rms = librosa.feature.rms(y=chunk, hop_length=hop)[0]
    energy_feats = [np.mean(rms), np.std(rms), np.max(rms), np.min(rms), np.max(rms)-np.min(rms)]
    
    # Duration (2 dims)
    dur = len(chunk) / 22050
    voiced_count = np.sum(rms > np.mean(rms))
    duration_feats = [dur, dur / (voiced_count + 1)]
    
    # Spectral (5 dims)
    try:
        sc = librosa.feature.spectral_centroid(y=chunk, sr=22050, hop_length=hop)[0]
        sb = librosa.feature.spectral_bandwidth(y=chunk, sr=22050, hop_length=hop)[0]
        sf2 = librosa.feature.spectral_flatness(y=chunk, hop_length=hop)[0]
        zcr = librosa.feature.zero_crossing_rate(chunk, hop_length=hop)[0]
        spectral_feats = [np.mean(sc), np.mean(sb), np.mean(sf2), np.mean(zcr), np.std(zcr)]
    except: spectral_feats = [0]*5
    
    # Voice quality (6 dims)
    try:
        yh, _ = librosa.effects.hpss(chunk)
        hnr = np.mean(np.abs(yh)) / (np.mean(np.abs(chunk)) + 1e-8)
        voice_feats = [hnr, np.mean(np.abs(chunk)), np.std(chunk), np.max(np.abs(chunk)), 0, 0]
    except: voice_feats = [0]*6
    
    return np.array(f0_feats + energy_feats + duration_feats + spectral_feats + voice_feats, dtype=np.float32)

def extract_word_features(audio_path, timestamps, labels):
    """Extract WavLM + prosody for all words."""
    y_16k = load_audio(audio_path)  # For WavLM
    y_22k = librosa.resample(y_16k, orig_sr=16000, target_sr=22050)  # For prosody
    
    n = len(timestamps)
    wavlm_feats = np.zeros((n, 768), dtype=np.float32)
    prosody_feats = np.zeros((n, 23), dtype=np.float32)
    
    for i, (t0, t1) in enumerate(timestamps):
        s, e = int(t0 * SR_AUDIO), min(int(t1 * SR_AUDIO), len(y_16k))
        chunk = y_16k[s:e]
        
        if len(chunk) < 0.05 * SR_AUDIO:
            continue
        
        # WavLM
        chunk_t = torch.tensor(chunk).unsqueeze(0).float().to(device)
        with torch.no_grad():
            rep = wavlm(chunk_t).last_hidden_state
        wavlm_feats[i] = rep.mean(dim=2).squeeze().cpu().numpy()
        
        # Prosody
        s22, e22 = int(t0 * 22050), min(int(t1 * 22050), len(y_22k))
        chunk22 = y_22k[s22:e22]
        pf = extract_23dim_prosody(y_22k, t0, t1)
        if pf is not None:
            prosody_feats[i] = pf
    
    # Combine
    combined = np.concatenate([wavlm_feats, prosody_feats], axis=1)  # (n, 791)
    return combined.astype(np.float32)

print('✓ Feature extractors ready')

In [ ]:
# Cell 4: Find all videos with EMNLP labels
import os

# Scan all language subsets
all_videos = []
for lang in ['en_uk', 'en_us', 'es', 'es_latam', 'fr', 'fr_ca', 'it', 'cs', 'hu', 'pt']:
    label_dir = f'{LABEL_BASE}/{lang}/emnlp+jahak/all'
    if not os.path.exists(label_dir): continue
    
    for fname in sorted(os.listdir(label_dir)):
        if not fname.endswith('.csv'): continue
        vid = fname.replace('.csv', '')
        all_videos.append((vid, lang))

print(f'Total videos with labels: {len(all_videos)}')

# Find checkpoint - which videos already done?
checkpoint_file = f'{FEAT_DIR}/checkpoint.json'
if os.path.exists(checkpoint_file):
    with open(checkpoint_file) as f:
        checkpoint = json.load(f)
    done_vids = set(checkpoint.get('done', []))
    print(f'Already extracted: {len(done_vids)}')
else:
    done_vids = set()
    checkpoint = {'done': []}

# Filter to pending
pending = [(v, l) for v, l in all_videos if v not in done_vids]
print(f'Pending: {len(pending)}')
print(f'Sample pending: {[v for v, _ in pending[:5]]}')

In [ ]:
# Cell 5: Extract features (auto-resume, checkpoint every 10 videos)
import time

def find_audio(vid):
    """Find audio file for video ID."""
    for audio_dir in [f'{BASE}/audio', f'{BASE}/audio_1000', f'{BASE}']:
        for ext in ['.wav', '.m4a', '.webm']:
            p = f'{audio_dir}/{vid}{ext}'
            if os.path.exists(p): return p
    return None

def save_checkpoint(done_vids):
    checkpoint['done'] = list(done_vids)
    with open(checkpoint_file, 'w') as f:
        json.dump(checkpoint, f)

t0 = time.time()
BATCH_SIZE = 10  # Save checkpoint every 10 videos

for i, (vid, lang) in enumerate(pending):
    # Find audio
    audio_path = find_audio(vid)
    if not audio_path:
        continue  # Skip if no audio
    
    # Load labels
    label_path = f'{LABEL_BASE}/{lang}/emnlp+jahak/all/{vid}.csv'
    if not os.path.exists(label_path): continue
    
    try:
        df = pd.read_csv(label_path)
        timestamps, text_labels = [], []
        for _, row in df.iterrows():
            try:
                ts = ast.literal_eval(str(row['timestamp']))
                timestamps.append((float(ts[0]), float(ts[1]))
            except: continue
        
        if len(timestamps) == 0: continue
        
        # Extract features
        features = extract_word_features(audio_path, timestamps, text_labels)
        
        # Save
        np.save(f'{FEAT_DIR}/{vid}_features.npy', features)
        np.save(f'{FEAT_DIR}/{vid}_labels.npy', np.array(text_labels))
        
        done_vids.add(vid)
        
        # Progress
        elapsed = time.time() - t0
        rate = (i + 1) / (elapsed / 60)  # videos per minute
        remaining = len(pending) - i - 1
        eta_min = remaining / rate if rate > 0 else 0
        
        print(f'[{i+1}/{len(pending)}] {vid}: {len(timestamps)} words | '
              f'Done: {len(done_vids)} | ETA: {eta_min:.0f} min | '
              f'{rate:.1f} vid/min', flush=True)
        
        # Checkpoint
        if (i + 1) % BATCH_SIZE == 0:
            save_checkpoint(done_vids)
            
    except Exception as e:
        print(f'ERROR {vid}: {str(e)[:80]}')
        continue

# Final save
save_checkpoint(done_vids)
print(f'
✓ COMPLETE: {len(done_vids)} videos extracted')
print(f'Total time: {(time.time()-t0)/3600:.1f} hours')

In [ ]:
# Cell 6: Summary
print('=== EXTRACTION COMPLETE ===')
done_vids = checkpoint['done']
print(f'Total videos: {len(done_vids)}')

# Count features
feature_files = [f for f in os.listdir(FEAT_DIR) if f.endswith('_features.npy')]
print(f'Feature files: {len(feature_files)}')

# Check feature shapes
for f in feature_files[:3]:
    vid = f.replace('_features.npy', '')
    feat = np.load(f'{FEAT_DIR}/{f}')
    labels = np.load(f'{FEAT_DIR}/{vid}_labels.npy') if os.path.exists(f'{FEAT_DIR}/{vid}_labels.npy') else None
    print(f'  {vid}: {feat.shape} features, {len(labels) if labels is not None else "?"} words')

print(f'
Features saved to: {FEAT_DIR}')
print('
Next: Run training notebook on extracted features.')